In [ ]:
!pip install llama-index llama-index-llms-openai streamlit pypdf pyngrok python-docx

In [ ]:
%%writefile app.py

import streamlit as st
import tempfile, os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

st.title("📂 Ask Your Documents")

uploaded_files = st.file_uploader(
    "Upload files (PDF, TXT, DOCX)",
    accept_multiple_files=True
)

if uploaded_files:
    with tempfile.TemporaryDirectory() as tmpdir:
        for f in uploaded_files:
            path = os.path.join(tmpdir, f.name)
            with open(path, "wb") as out:
                out.write(f.read())

        with st.spinner("Reading your files..."):
            docs = SimpleDirectoryReader(tmpdir).load_data()
            index = VectorStoreIndex.from_documents(docs)
            st.session_state.engine = index.as_query_engine()

        st.success(f"✅ Loaded {len(uploaded_files)} file(s)!")

question = st.text_input("Ask a question about your documents")

if st.button("Ask") and question:
    if "engine" in st.session_state:
        with st.spinner("Thinking..."):
            answer = st.session_state.engine.query(question)
            st.markdown(f"**Answer:** {answer}")
    else:
        st.warning("Please upload files first!")

Writing app.py


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "set OpenAI api key here"#set OpenAI api key here

In [ ]:
import subprocess, time
from pyngrok import ngrok

ngrok.set_auth_token("Set ngrok Api key here")#set ngrok api key here

subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

process = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
    "--server.address", "0.0.0.0"   # ← this is the key fix
])

time.sleep(5)
tunnel = ngrok.connect(8501, bind_tls=True)
print(f"✅ Open this URL: {tunnel.public_url}")

✅ Open this URL: https://power-imposing-lethargic.ngrok-free.dev
